# Spam — Seed Strategy Comparison

Compares three generation strategies side-by-side:

| Variant | Seed | Description |
|---|---|---|
| **HAM** | Real HAM SMS | Inject spam signals into legitimate messages |
| **SPAM** | Real SPAM SMS | Rewrite existing spam into variations |
| **Profile** | None | LLM describes the dataset, generates from scratch |

In [ ]:
import sys, os, json, glob
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

RUNS_DIR = os.path.join(project_root, 'framework', 'data', 'runs')
COLORS   = {'ham': '#3498db', 'spam': '#e67e22', 'profile': '#9b59b6'}
METRICS  = ['f1', 'precision', 'recall', 'accuracy']

def latest_session(variant):
    folder = os.path.join(RUNS_DIR, f'spam_{variant}')
    sessions = sorted(glob.glob(os.path.join(folder, '*')), reverse=True)
    return sessions[0] if sessions else None

def load_results(session_dir):
    with open(os.path.join(session_dir, 'results.json'), encoding='utf-8') as f:
        return json.load(f)

def load_generated(session_dir):
    items = []
    for path in sorted(glob.glob(os.path.join(session_dir, 'generated', 'run_*.json'))):
        with open(path, encoding='utf-8', errors='replace') as f:
            items.extend(json.load(f))
    return items

# Load all available variants
runs = {}
for v in ['ham', 'spam', 'profile']:
    session = latest_session(v)
    if session:
        results   = load_results(session)
        generated = load_generated(session)
        meta      = results.get('meta', {})
        runs[v]   = {'session': session, 'results': results, 'generated': generated, 'meta': meta}
        print(f"[{v:7}]  {os.path.basename(session)}  |  model: {meta.get('model','?')}  |  dataset: {meta.get('dataset',{}).get('name') or meta.get('dataset',{}).get('path','?')}  |  n={len(generated)}")
    else:
        print(f'[{v:7}]  no session found — skipping')

available = list(runs.keys())
models    = list(next(iter(runs.values()))['results']['results'].keys())
short     = {m: m.split('/')[-1] for m in models}
print(f'\nModels: {[short[m] for m in models]}')

## 1 · F1 Score Overview (Generated vs Real Baseline)

In [ ]:
def gen_score(results, model, metric):
    v = results['results'].get(model, {}).get('generated', {}).get(metric, {})
    return v.get('mean', float('nan')) if isinstance(v, dict) else float('nan')

def real_score(results, model, metric):
    v = results['results'].get(model, {}).get('real', {}).get(metric, float('nan'))
    return v if isinstance(v, (int, float)) else float('nan')

# Summary table: F1 generated per variant + real baseline (same across variants)
rows = []
for m in models:
    row = {'model': short[m]}
    for v in available:
        row[f'{v}_gen'] = gen_score(runs[v]['results'], m, 'f1')
    row['real_f1'] = real_score(runs[available[0]]['results'], m, 'f1')
    rows.append(row)

df = pd.DataFrame(rows).set_index('model')
df.columns = [c.replace('_gen', ' (gen)') for c in df.columns]
df.columns = [c.replace('real_f1', 'Real baseline') for c in df.columns]
print(df.round(3).to_string())

In [ ]:
# F1 chart: generated per variant + real baseline as dashed line
n_v   = len(available)
width = 0.7 / n_v
x     = np.arange(len(models))

fig, ax = plt.subplots(figsize=(10, 4.5))
for i, v in enumerate(available):
    offset = (i - (n_v - 1) / 2) * width
    vals   = [gen_score(runs[v]['results'], m, 'f1') for m in models]
    bars   = ax.bar(x + offset, vals, width * 0.92, label=v, color=COLORS[v])
    ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=7)

# Real baseline as scatter dots (same for all variants)
real_vals = [real_score(runs[available[0]]['results'], m, 'f1') for m in models]
ax.scatter(x, real_vals, marker='D', s=50, color='black', zorder=5, label='Real baseline')

ax.set_xticks(x)
ax.set_xticklabels([short[m] for m in models], rotation=20, ha='right', fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('F1')
ax.set_title('F1 on Synthetic Data by Seed Strategy  (◆ = real baseline)')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=False)
fig.tight_layout()
plt.savefig('spam_seed_comparison_scores.png', dpi=150)
plt.show()

## 2 · Spam Technique Distribution

In [ ]:
# Techniques are comma-separated strings — split and count individual techniques
def count_techniques(generated):
    c = Counter()
    for s in generated:
        raw = s.get('technique') or s.get('error_type') or ''
        for t in [t.strip() for t in raw.split(',') if t.strip()]:
            c[t] += 1
    return c

technique_counts = {v: count_techniques(runs[v]['generated']) for v in available}
all_techniques   = sorted({t for c in technique_counts.values() for t in c})

n_v   = len(available)
width = 0.7 / n_v
x     = np.arange(len(all_techniques))

fig, ax = plt.subplots(figsize=(max(8, len(all_techniques) * 1.5), 4.5))
for i, v in enumerate(available):
    offset = (i - (n_v - 1) / 2) * width
    vals   = [technique_counts[v].get(t, 0) for t in all_techniques]
    bars   = ax.bar(x + offset, vals, width * 0.92, label=v, color=COLORS[v])
    ax.bar_label(bars, fmt='%d', padding=2, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(all_techniques, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Count (SPAM-labeled samples)')
ax.set_title('Spam Technique Distribution by Seed Strategy')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=False)
fig.tight_layout()
plt.savefig('spam_seed_comparison_error_types.png', dpi=150)
plt.show()

## 3 · Label Distribution

In [ ]:
label_classes = ['SPAM', 'HAM']
n_v   = len(available)
width = 0.7 / n_v
x     = np.arange(len(label_classes))

fig, ax = plt.subplots(figsize=(6, 4))
for i, v in enumerate(available):
    total  = max(len(runs[v]['generated']), 1)
    counts = Counter(s.get('label') for s in runs[v]['generated'])
    props  = [counts.get(l, 0) / total for l in label_classes]
    offset = (i - (n_v - 1) / 2) * width
    bars   = ax.bar(x + offset, props, width * 0.92, label=v, color=COLORS[v])
    ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(label_classes)
ax.set_ylabel('Proportion')
ax.set_title('Label Distribution by Seed Strategy')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=False)
fig.tight_layout()
plt.savefig('spam_seed_comparison_labels.png', dpi=150)
plt.show()

## 4 · Text Length Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for v in available:
    lengths = [len(s.get('text', '').split()) for s in runs[v]['generated']]
    print(f'{v:7}  mean={np.mean(lengths):.1f}  median={np.median(lengths):.1f}  max={max(lengths)}')
    ax.hist(lengths, bins=30, alpha=0.5, label=v, color=COLORS[v])

ax.set_xlabel('Word count')
ax.set_ylabel('Frequency')
ax.set_title('Text Length Distribution by Seed Strategy')
ax.legend(frameon=False)
fig.tight_layout()
plt.savefig('spam_seed_comparison_length.png', dpi=150)
plt.show()

## 5 · Sample Examples

In [ ]:
for v in available:
    generated = runs[v]['generated']
    print(f'\n{"="*60}')
    print(f'  {v.upper()} — SPAM samples')
    print(f'{"="*60}')
    for s in [x for x in generated if x.get('label') == 'SPAM'][:3]:
        tech = s.get('technique') or s.get('error_type') or '-'
        print(f'  [{tech}]')
        print(f'  {s.get("text","")[:150]}\n')
    print(f'  --- HAM samples ---')
    for s in [x for x in generated if x.get('label') == 'HAM'][:2]:
        print(f'  {s.get("text","")[:150]}\n')

## 6 · Dataset Profile (Profile Variant)

In [ ]:
if 'profile' in runs:
    text = runs['profile']['meta'].get('dataset_profile')
    if text:
        print(text)
    else:
        print('No dataset_profile in meta.')
else:
    print('Profile variant not run.')